# Simple LSTM with Tiny Shakespeare

In [11]:
import tensorflow as tf
import tensorflow_datasets as tfds

# 1. Load Data
# We treat the text as one long string initially
dataset = tfds.load('tiny_shakespeare', split='train', as_supervised=False)
# Get the raw text from the dataset
one_example = next(iter(dataset))
text = one_example['text'].numpy().decode('utf-8')

# 2. Vectorize (Create the Vocabulary)
vocab = sorted(set(text))
vocab_size = len(vocab)
print(f'Vocabulary size: {vocab_size}')

# Create the mapping layers
ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)

# Convert all text to IDs at once
all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)

# 3. Create Sequences
seq_length = 100  # Length of the sequence to train on
sequences = ids_dataset.batch(seq_length + 1, drop_remainder=True)

# 4. Split into Input and Target
# If sequence is "Hello", input="Hell", target="ello"
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

# 5. Batch and Shuffle (Performance Optimization)
BATCH_SIZE = 64
BUFFER_SIZE = 10000 # For shuffling

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE)
)

# --- MODEL ---
model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(len(ids_from_chars.get_vocabulary()), 256),
    tf.keras.layers.LSTM(1024, return_sequences=True),
    tf.keras.layers.Dense(len(ids_from_chars.get_vocabulary()))
])

# Compile
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam', loss=loss, metrics=['accuracy'])

print("Data ready. Model compiled.")

Vocabulary size: 65
Data ready. Model compiled.


In [12]:
history = model.fit(dataset, epochs=10)

Epoch 1/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 260s 2s/step - accuracy: 0.2705 - loss: 2.7090
Epoch 2/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 255s 2s/step - accuracy: 0.4014 - loss: 2.0515
Epoch 3/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 253s 2s/step - accuracy: 0.4698 - loss: 1.7966
Epoch 4/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 266s 2s/step - accuracy: 0.5097 - loss: 1.6431
Epoch 5/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 271s 2s/step - accuracy: 0.5351 - loss: 1.5462
Epoch 6/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 273s 2s/step - accuracy: 0.5524 - loss: 1.4784
Epoch 7/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 281s 2s/step - accuracy: 0.5654 - loss: 1.4275
Epoch 8/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 271s 2s/step - accuracy: 0.5756 - loss: 1.3876
Epoch 9/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 269s 2s/step - accuracy: 0.5839 - loss: 1.3546
Epoch 10/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 267s 2s/step - accuracy: 0.5910 - loss: 1.3259


In [ ]:
def generate_text(model, start_string, num_generate=1000, temperature=1.0):
    # 1. Convert the start string to numbers (Vectorize)
    input_ids = ids_from_chars(tf.strings.unicode_split(start_string, 'UTF-8'))
    input_ids = tf.expand_dims(input_ids, 0) # Add batch dimension

    text_generated = []

    # 2. The Generation Loop
    model.reset_states() # Clear any hidden state from training
    for i in range(num_generate):
        predictions = model(input_ids)
        
        # Remove the batch dimension
        predictions = tf.squeeze(predictions, 0)

        # 3. Apply Temperature (Higher = More Random)
        predictions = predictions / temperature
        
        # 4. Sample from the distribution
        # We don't use argmax! We pick based on probability.
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

        # 5. Feed the predicted letter back in as the next input
        input_ids = tf.expand_dims([predicted_id], 0)
        
        # Store the result
        text_generated.append(chars_from_ids(predicted_id))

    return (start_string + tf.strings.reduce_join(text_generated).numpy().decode("utf-8"))


In [ ]:

# --- Run it! ---
print(generate_text(model, start_string=u"ROMEO: ", temperature=1.0))